In [ ]:
import os
import streamlit as st
import pickle
import time
import langchain
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import UnstructuredURLLoader
import faiss
from sentence_transformers import SentenceTransformer

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

### load data by **UnstructuredURLLoader**

In [3]:
loaders = UnstructuredURLLoader(urls=[
    "https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html",
    "https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html"
])
data = loaders.load() 
len(data)

2

### text spliting to create chunks by **RecursiveCharacterTextSplitter**

In [9]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(data)
print("Number of chunks:", len(chunks))
chunks[0]

Number of chunks: 18


Document(metadata={'source': 'https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html'}, page_content='English\n\nHindi\n\nGujarati\n\nSpecials\n\nMy Alerts\n\nGo Ad-Free\n\nHello, Login\n\nHello, Login\n\nLog-inor Sign-Up\n\nMy Account\n\nMy Profile\n\nMy Portfolio\n\nMy Watchlist\n\nMy Alerts\n\nMy Messages\n\nPrice Alerts\n\nMy Profile\n\nMy PRO\n\nMy Portfolio\n\nMy Watchlist\n\nMy Alerts\n\nMy Messages\n\nPrice Alerts\n\nLogout\n\nLoans up to ₹50 LAKHS\n\nFixed Deposits\n\nCredit CardsLifetime Free\n\nCredit Score\n\nLoan against MFs\n\nChat with Us\n\nDownload App\n\nFollow us on:\n\nNetwork 18\n\n>->MC_ENG_DESKTOP/MC_ENG_NEWS/MC_ENG_MARKETS_AS/MC_ENG_ROS_NWS_MKTS_AS_ATF_728\n\nMoneycontrol\n\nGo PRO NowPRO\n\nMoneycontrol PRO\n\nAdvertisement\n\nRemove Ad\n\nBusiness\n\nMarkets\n\nStocks\n\nEconomy\n\nCompanies\n\nTrends\n\nIPO\n\nOpinion\n\nEV Special\n\nEco Pulse\n\nMC Learn\n\nArray\n(\n    [direction] => 1\n    [market

### create embeddings by **OpenAIEmbeddings** and save them to **FAISS** index

In [11]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"  # Small, fast, free
)

vector_index = FAISS.from_documents(chunks, embeddings)

file_path = "vector_index.pkl"

with open(file_path, "wb") as f:
    pickle.dump(vector_index, f)

C:\Users\Ali Akbar\AppData\Local\Temp\ipykernel_11480\3929757522.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
c:\Users\Ali Akbar\anaconda3\envs\ml\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ali Akbar\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable.

### retrieve similar chunks from the texts, combine them and **retrieve final answer**

**User Query → Retrieve Docs → Add Context → LLM → Answer**

```
Query  
 ↓  
Retriever → Relevant chunks  
 ↓  
Format → Clean text  
 ↓  
Prompt → Structured input  
 ↓  
LLM → Generate answer  
 ↓  
Parser → Final string  
```

In [25]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [29]:

load_dotenv()  # Load environment variables from .env file
api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7,
    api_key=api_key
)

In [30]:
# retriever - chunks from FAISS vector store...

retriever = vector_index.as_retriever()

# prompt
prompt = ChatPromptTemplate.from_template(
    """
Answer the question based only on the context.

Context:
{context}

Question:
{question}
"""
)

# format docs
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

# chain (modern LCEL)
rag_chain = (
    {
        # The "format_docs" function isn't being called immediately 
        # - it's being passed as a reference to be called later by the chain.

        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# or we can use this type of code. (No RunnablePassthrough needed)

# chain = (
#     {"context": retriever}  # Input must be dict with 'input' key
#     | prompt
#     | llm
# )
# result = chain.invoke({"input": "what is the price of Tiago iCNG?"})

# run
query = "what is the price of Tiago iCNG?"

result = rag_chain.invoke(query)

print(result)

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "input": "what is the price of Tiago iCNG?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question>] Entering Chain run with input:
{
  "input": "what is the price of Tiago iCNG?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question> > chain:RunnableSequence] Entering Chain run with input:
{
  "input": "what is the price of Tiago iCNG?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question> > chain:RunnableSequence > chain:format_docs] Entering Chain run with input:
[inputs]
[chain/end] [chain:RunnableSequence > chain:RunnableParallel<context,question> > chain:RunnableSequence > chain:format_docs] s] Exiting Chain run with output:
{
  "output": "The company also said it has also introduced the twin-cylinder technology on its Tiago and Tigor models.\n\nThe Tiago iCNG is priced between Rs 6.55 lakh and Rs 8.1 lakh, while the Tigor i

### retrieval with **sources**

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Original format_docs
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

# Function to extract sources from docs
def extract_sources(docs):
    sources = []
    for doc in docs:
        # Try different metadata keys where URL might be stored
        url = (doc.metadata.get('url') or 
               doc.metadata.get('source') or 
               doc.metadata.get('link') or
               doc.metadata.get('URL'))
        if url:
            sources.append(url)
    return list(set(sources))  # Remove duplicates

retriever = vector_index.as_retriever()

prompt = ChatPromptTemplate.from_template(
    """
Answer the question based only on the context.

Context:
{context}

Question:
{question}
"""
)

# Create the chain (for answer only)
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# Run and get sources separately
query = "what is the Currency of Bangladesh?"

# Get the retrieved documents
retrieved_docs = retriever.invoke(query)

# Get the answer
answer = rag_chain.invoke(query)

# Extract sources
sources = extract_sources(retrieved_docs)

# Display results
print(f"Answer: {answer}\n")
print(f"Sources:")
for i, source in enumerate(sources, 1):
    print(f"{i}. {source}")

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "input": "what is the Currency of Bangladesh?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question>] Entering Chain run with input:
{
  "input": "what is the Currency of Bangladesh?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question> > chain:RunnableSequence] Entering Chain run with input:
{
  "input": "what is the Currency of Bangladesh?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question> > chain:RunnableSequence > chain:format_docs] Entering Chain run with input:
[inputs][chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question> > chain:RunnablePassthrough] Entering Chain run with input:
{
  "input": "what is the Currency of Bangladesh?"
}
[chain/end] [chain:RunnableSequence > chain:RunnableParallel<context,question> > chain:RunnablePassthrough] s] Exiting Chain run with output:
{
  "output": "wha